# Day 008：GQA 与共享 K/V Head

本 Notebook 与 [Day 008 互动档案](../day-008.md) 配套。它从‘每个 token 拥有几个 Q/K/V’开始，依次运行 K/V Head 对齐、完整简化 GQA、参数量比较，并在本地模型存在时检查 MiniMind-3 的真实投影权重。

当前只讨论 GQA。RoPE 和 KV Cache 的内部生成流程仍留到后续学习日。

## 1. 先固定 token、Head 和向量之间的关系

最小配置有 2 个 token（‘吃了’、‘苹果’）、2 个 Q Head、1 个 K Head、1 个 V Head，每个向量有 2 个数。每个 token 因此拥有 2 个 Q 向量、1 个 K 向量和 1 个 V 向量。

统一使用 `[batch, heads, sequence, head_dim]` 排列张量。

In [ ]:
import torch

torch.set_printoptions(precision=4, sci_mode=False)

tokens = ["吃了", "苹果"]
head_dim = 2

Q = torch.tensor([[[[0.0, 1.0], [1.0, 2.0]],
                   [[1.0, 0.0], [3.0, 1.0]]]])
K = torch.tensor([[[[2.0, 0.0], [0.0, 2.0]]]])
V = torch.tensor([[[[10.0, 0.0], [0.0, 10.0]]]])

print("Q shape：", Q.shape)
print("K shape：", K.shape)
print("V shape：", V.shape)
print("苹果的两个 Q：")
print("Q0_苹果：", Q[0, 0, 1])
print("Q1_苹果：", Q[0, 1, 1])
print("共享 K Head 中两个 token 的 K：")
print("K0_吃了：", K[0, 0, 0])
print("K0_苹果：", K[0, 0, 1])

这里的‘1 个 K Head’不是整句话只有一个 K 向量。K Head 中仍然包含每个 token 位置自己的 K；V Head 同理。两个 Q Head 共享的是同一组 K/V 信息来源。

## 2. `repeat_kv` 只对齐 Head 维

Q 有 2 个 Head，K/V 只有 1 个 Head。为了用一次批量矩阵乘法完成两个 Q Head 的计算，把 K/V 在 Head 维对齐到 2。这里使用 `expand` 表达共享关系；它不会创建新的投影层或学习规则。

In [ ]:
def repeat_kv(hidden_states, repeats):
    batch, num_kv_heads, sequence, dim = hidden_states.shape
    if repeats == 1:
        return hidden_states
    return (
        hidden_states[:, :, None, :, :]
        .expand(batch, num_kv_heads, repeats, sequence, dim)
        .reshape(batch, num_kv_heads * repeats, sequence, dim)
    )

K_aligned = repeat_kv(K, repeats=2)
V_aligned = repeat_kv(V, repeats=2)

print("重复前 K shape：", K.shape)
print("重复后 K shape：", K_aligned.shape)
print("重复前 V shape：", V.shape)
print("重复后 V shape：", V_aligned.shape)
print("两个 K Head 内容是否相同：", torch.equal(K_aligned[:, 0], K_aligned[:, 1]))
print("两个 V Head 内容是否相同：", torch.equal(V_aligned[:, 0], V_aligned[:, 1]))

## 3. 两个 Q Head 独立完成 Attention

K/V 虽然相同，但两个 Q Head 的 Q 不同。因此它们会分别产生自己的匹配分数、Softmax 读取比例和 Head 输出。

In [ ]:
scores = (Q @ K_aligned.transpose(-2, -1)) / (head_dim ** 0.5)

sequence = Q.shape[2]
causal_mask = torch.triu(
    torch.ones(sequence, sequence, dtype=torch.bool),
    diagonal=1
)
masked_scores = scores.masked_fill(causal_mask, float("-inf"))
weights = torch.softmax(masked_scores, dim=-1)
head_results = weights @ V_aligned

print("scores shape：", scores.shape)
print("weights shape：", weights.shape)
print("head_results shape：", head_results.shape)
print("两个 Q Head 的匹配分数：")
print(scores)
print("两个 Q Head 的读取比例：")
print(weights)
print("每行比例之和：")
print(weights.sum(dim=-1))
print("两个 Q Head 的读取结果：")
print(head_results)
print("苹果在 Q Head 0 中的读取结果：", head_results[0, 0, 1])
print("苹果在 Q Head 1 中的读取结果：", head_results[0, 1, 1])

实验结论：

```text
共享相同 K/V + 使用不同 Q
-> 不同匹配分数
-> 不同 Softmax 比例
-> 不同 Head 输出
```

所以 GQA 共享的是 K/V 信息来源，不是整个 Attention 计算结果。

## 4. 映射到 MiniMind-3 的 shape

MiniMind-3 使用 `hidden_size=768`、8 个 Q Head、4 个 K/V Head、`head_dim=96`。

In [ ]:
batch = 1
sequence = 11
hidden_size = 768
num_q_heads = 8
num_kv_heads = 4
head_dim = 96

q_width = num_q_heads * head_dim
kv_width = num_kv_heads * head_dim

print("h：", (batch, sequence, hidden_size))
print("q_projected：", (batch, sequence, q_width))
print("k/v_projected：", (batch, sequence, kv_width))
print("Q Head 形式：", (batch, num_q_heads, sequence, head_dim))
print("原始 K/V Head 形式：", (batch, num_kv_heads, sequence, head_dim))
print("对齐后的 K/V：", (batch, num_q_heads, sequence, head_dim))
print("scores/weights：", (batch, num_q_heads, sequence, sequence))
print("head_results：", (batch, num_q_heads, sequence, head_dim))
print("combined：", (batch, sequence, num_q_heads * head_dim))

## 5. 小模型中比较 MHA 与 GQA

继续使用 Day 007 的 `hidden_size=8`、2 个 Q Head、`head_dim=4`。MHA 有 2 个 K/V Head，GQA 只有 1 个 K/V Head。所有 Linear 都设置 `bias=False`。

In [ ]:
from torch import nn

hidden_size = 8
num_q_heads = 2
mha_num_kv_heads = 2
gqa_num_kv_heads = 1
head_dim = 4

mha_q = nn.Linear(hidden_size, num_q_heads * head_dim, bias=False)
mha_k = nn.Linear(hidden_size, mha_num_kv_heads * head_dim, bias=False)
mha_v = nn.Linear(hidden_size, mha_num_kv_heads * head_dim, bias=False)
gqa_q = nn.Linear(hidden_size, num_q_heads * head_dim, bias=False)
gqa_k = nn.Linear(hidden_size, gqa_num_kv_heads * head_dim, bias=False)
gqa_v = nn.Linear(hidden_size, gqa_num_kv_heads * head_dim, bias=False)

def parameter_count(*layers):
    return sum(parameter.numel() for layer in layers for parameter in layer.parameters())

print("MHA q/k/v 参数：", [parameter_count(layer) for layer in (mha_q, mha_k, mha_v)])
print("GQA q/k/v 参数：", [parameter_count(layer) for layer in (gqa_q, gqa_k, gqa_v)])
print("MHA Q/K/V 总参数：", parameter_count(mha_q, mha_k, mha_v))
print("GQA Q/K/V 总参数：", parameter_count(gqa_q, gqa_k, gqa_v))

batch = 1
sequence = 3
mha_kv_values = batch * sequence * (mha_num_kv_heads * head_dim) * 2
gqa_kv_values = batch * sequence * (gqa_num_kv_heads * head_dim) * 2
print("3 个 token 的 MHA K/V 数值总量：", mha_kv_values)
print("3 个 token 的 GQA K/V 数值总量：", gqa_kv_values)
print("每个新 token 的 MHA KV Cache 增量：", mha_num_kv_heads * head_dim * 2)
print("每个新 token 的 GQA KV Cache 增量：", gqa_num_kv_heads * head_dim * 2)

这里要区分三件事：

- 参数量：投影层权重中长期保存多少个可学习数字。
- 运行时 K/V 数据量：本次输入经过投影后产生多少个数字。
- KV Cache：生成过程中为旧 token 保存多少 K/V；这里只比较大小，不展开缓存流程。

## 6. 检查 MiniMind-3 的真实权重

下面的 Cell 会从常见相对位置寻找本地 `minimind-3`。找到模型时加载并检查第一层 Attention；没有本地模型时只跳过这一项，不影响前面的简化实验。

In [ ]:
from pathlib import Path

search_roots = [Path.cwd(), *Path.cwd().parents]
model_candidates = []
for root in search_roots:
    model_candidates.extend([root / "minimind-3", root / "minimind" / "minimind-3"])
model_dir = next((path for path in model_candidates if (path / "config.json").is_file()), None)

if model_dir is None:
    print("未找到本地 minimind-3，跳过真实权重检查。")
else:
    from transformers import AutoModelForCausalLM

    print("加载本地模型：", model_dir)
    real_model = AutoModelForCausalLM.from_pretrained(model_dir, local_files_only=True)
    attention = real_model.model.layers[0].self_attn
    real_counts = {}
    for name in ["q_proj", "k_proj", "v_proj", "o_proj"]:
        projection = getattr(attention, name)
        real_counts[name] = projection.weight.numel()
        print(
            name,
            "weight shape =", projection.weight.shape,
            "参数量 =", projection.weight.numel(),
            "bias =", projection.bias,
        )

    gqa_qkv = sum(real_counts[name] for name in ["q_proj", "k_proj", "v_proj"])
    gqa_qkvo = gqa_qkv + real_counts["o_proj"]
    mha_qkv = real_counts["q_proj"] * 3
    mha_qkvo = mha_qkv + real_counts["o_proj"]
    print("GQA Q/K/V：", gqa_qkv)
    print("假设改为 MHA 的 Q/K/V：", mha_qkv)
    print("GQA Q/K/V/O：", gqa_qkvo)
    print("假设改为 MHA 的 Q/K/V/O：", mha_qkvo)
    print("Q/K/V/O 节省比例：", (mha_qkvo - gqa_qkvo) / mha_qkvo)


## 7. 完整结论

MiniMind-3 把 8 个 Q Head 分成 4 组，每两个 Q Head 共享 1 个 K Head 和 1 个 V Head。Q、匹配分数、Softmax 比例和 Head 输出仍各自独立。

相比同规模 MHA，GQA 的收益是 K/V 投影参数更少、K/V 投影计算更少、KV Cache 更小；代价是可独立学习的 K/V 投影规则更少。MiniMind-3 的 Q/K/V 投影参数减少三分之一；计入不变的 `o_proj` 后，Q/K/V/O 四个投影合计减少四分之一。